<p> <center><img src="https://datascientest.fr/train/assets/logo_datascientest.png" style="height:200px"></center> </p>

<hr style="border-width:2px;border-color:#ff6745">
<center><h1> Analyse et prédiction de la variabilité de la production solaire à partir de données ouvertes de la région PACA </h1></center>
<center><h2> Collecte des données atmosphériques </h2></center>
<hr style="border-width:2px;border-color:#ff6745">


Nous avons précédemment collecté les données de **production d'énergie solaire** et les données sur la **position du soleil**. Or, parmi les connaissances de l'homme du métier, d'autres données que celles concernant la position du soleil pourraient permettre d'expliquer notre variable cible et être utiles à notre modélisation.

Ainsi, dans ce notebook, nous nous intéressons aux *rayonnements solaires effectivements reçus* au niveau des panneaux solaires et donc à des données concernant l'**atmosphère**.

Ces données sont par exemple disponibles dans un jeu de données `CAMS solar radiation time-serie` mis gracieusement à disposition par Copernicus (programme d’observation de la Terre de l’Union européenne - https://www.copernicus.eu/fr) via son service de surveillance de l’atmosphère (CAMS - https://atmosphere.copernicus.eu/), sous licence CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/legalcode vu sur https://ads.atmosphere.copernicus.eu/datasets/cams-solar-radiation-timeseries?tab=overview).

Pour chacun des points d'intérêts identifiés pour la production d'énergie solaire, nous récupèrerons les variables :
 - **GHI** : irradiance horizontale globale *(mesure globale de la lumière solaire reçue sur un plan horizontal au niveau du sol)* ;
 - **BHI** : irradiance horizontale directe *(mesure des rayonnements solaires directs reçus sur un plan horizontal au niveau du sol)* ;
 - **DHI** : irradiance horizontale diffuse *(mesure des rayonnements solaires indirects, diffusés dans l'atmosphère reçus sur un plan horizontal au niveau du sol)* ;
 - **BNI** : irradiance normale directe *(mesure des rayonnements solaires directs reçus sur un plan perpendiculaire aux rayons solaires et se déplaçant avec ce dernier)*.

Pour chacune de ces variables, nous récupèreront également leur valeur théorique maximale (pour un ciel dégagé).


# I - Récupération des données des étapes précédentes


Nous commencons par importer les librairies nécessaires pour manipuler nos données :


In [1]:
# Gestion des chemins
from pathlib import Path

# On importe les librairies dont on a besoin pour traiter les datasets
import pandas as pd
import numpy as np

Puis on récupère les données collectées aux étapes précédentes :

In [2]:
# Chemin vers le répertoire de données d'entrée
input_path = Path('../../data/local_data/input/')

# Chemin vers le répertoire de résultats temporaires
temp_path = Path('../../data/local_data/temp/')

# Chemin vers le répertoire de résultats finaux
output_path = Path('../../data/local_data/output/')

# Chemin du dataset de production
input_datasets = temp_path / 'production_2020_2025.csv'

# Chemin du dataset des communes sélectionnées 
communes = output_path / 'best_communes_geo_energy.csv'

On charge le jeu de données de l'étape précédente :

In [3]:
# On récupère le dataset contenant les données de production
df_atmo = pd.read_csv(input_datasets, parse_dates=['datetime_utc'])

# Pour la collecte des données on n'a besoin que des créneaux horaires
df_atmo = df_atmo[['datetime_utc']].copy()
display(df_atmo.head())


,datetime_utc
0,2019-12-31 23:00:00+00:00
1,2019-12-31 23:30:00+00:00
2,2020-01-01 00:00:00+00:00
3,2020-01-01 00:30:00+00:00
4,2020-01-01 01:00:00+00:00


# II - Lieux retenus

On récupère les coordonnées des cinq points significatifs que l'on a précédemment déterminé.


In [4]:
df_communes = pd.read_csv(communes)
display(df_communes.head())
print(df_communes.dtypes)

,cluster_geo,best_commune,code_insee,lat,lon,energie_totale,poids,prefix
0,2,Cruis,4065,44.0845,5.8397,20356525.0,0.22,cru
1,4,Saint-Étienne-le-Laus,5140,44.5075,6.1616,325158.0,0.06,sel
2,0,Saint-Vallier-de-Thiey,6130,43.6994,6.8516,344281.0,0.07,svt
3,1,Bras,83021,43.4723,5.9558,10603661.0,0.29,bra
4,3,Eygalières,13034,43.7638,4.9554,1510927.0,0.36,eyg


cluster_geo         int64
best_commune       object
code_insee          int64
lat               float64
lon               float64
energie_totale    float64
poids             float64
prefix             object
dtype: object


# III - Collecte des données atmosphériques

Le jeu de données **CAMS solar radiation time-serie** (https://ads.atmosphere.copernicus.eu/datasets/cams-solar-radiation-timeseries?tab=overview) est accessible via une API par l'intermédiaire de la librairie cdsapi.


On importe cdsapi avec d'autres librairies utiles pour la collecte :

In [5]:
# On importe les librairies dont on a besoin pour accéder aux données
import yaml
import os
import cdsapi
import time

L'accès à l'API se fait via un compte : on charge donc les données correspondantes pour accéder à l'API :

In [6]:
# URL de l'API
url = "https://ads.atmosphere.copernicus.eu/api"

# API Token
with open(input_path / 'cams_access', 'r') as f:
    key = f.read().splitlines()[0] # On lit la première ligne du fichier qui contient une clé


Pour faciliter la collecte des données pour chacun de nos points d'intérêts, on crée une fonction qui interroge l'API en fonction des coordonnées géographiques d'un lieu (latitude et longitude) et des dates de début et de fin de la période à observer.

Un jeu de données est alors enregistré au niveau d'un chemin transmis à la fonction.


In [7]:
def retrieve_ghi (latitude, longitude, date_debut, date_fin, base_chemin) :
    # On crée le client de l'API
    client = cdsapi.Client(url=url, key=key)
    
    # Nom du jeu de données qui nous intéresse
    dataset = "cams-solar-radiation-timeseries"
    
    # Formulation de la requête
    requete = {"sky_type": "observed_cloud",
               "location": {"longitude": longitude, "latitude": latitude},
               "altitude": ["0"],
               "date": [date_debut + "/" + date_fin],
               "time_step": "15minute", # '30minute' n'existe pas on passe directement à un intervalle d'une heure
               "time_reference": "universal_time",
               "data_format": "csv"}
    
    target = base_chemin.parent / (base_chemin.name + "_" + date_debut + "_" + date_fin + ".csv")
    
    # Envoi de la requête
    fname = client.retrieve(dataset, requete, target)
    

On détermine, à partir du jeu de données précédemment collecté, la date de début et la date de fin de la période couverte.

In [8]:
# On détermine la plage de dates à requêter
# En heures UTC le dataset de départ commence le 31 décembre 2019 et se termine le 31 décembre 2024

date_debut = df_atmo.loc[0,'datetime_utc'].strftime('%Y-%m-%d')
date_fin = df_atmo.loc[df_atmo.shape[0]-1,'datetime_utc'].strftime('%Y-%m-%d')

print("Date de début :", date_debut)
print("Date de fin :", date_fin)

Date de début : 2019-12-31
Date de fin : 2026-01-31


On interroge l'API pour collecter les données atmosphérique pour chaque point d'intérêt :

In [9]:
# On interroge l'API pour récupérer les données CAMS de chaque ville
# Pour chaque ville
for ville in df_communes['prefix'] :
    # On affiche où on en est rendu dans les villes
    print(ville + "...")
    
    # Le chemin de base du dataset temporaire sera :
    base_chemin = temp_path / ("cams_" + ville) # La fonction de récupération ajoutera les dates et l'extension

    latitude = df_communes.loc[df_communes['prefix']==ville, 'lat'].iloc[0]
    longitude = df_communes.loc[df_communes['prefix']==ville, 'lon'].iloc[0]
    
    # Envoi de la requete
    retrieve_ghi(
        latitude, longitude,
        date_debut, date_fin,
        base_chemin)
    
    # On indique que la requete est terminée
    print("Ok pour", ville, '\n')
    
    # On patiente avant la requête suivante
    time.sleep(10)


cru...


2026-05-22 12:03:22,573 INFO Request ID is eac633cd-f4bd-400a-9fb4-a3d914cd5699
2026-05-22 12:03:22,658 INFO status has been updated to accepted
2026-05-22 12:03:44,002 INFO status has been updated to running
2026-05-22 12:04:38,460 INFO status has been updated to successful


Ok pour cru 

sel...


2026-05-22 12:04:54,587 INFO Request ID is ce4d0b27-1ed7-4448-8bfe-35d168e7f887
2026-05-22 12:04:54,657 INFO status has been updated to accepted
2026-05-22 12:05:15,884 INFO status has been updated to running
2026-05-22 12:06:10,511 INFO status has been updated to successful


Ok pour sel 

svt...


...

Ok pour svt 

bra...


2026-05-22 12:07:33,855 INFO Request ID is 486023cf-4fef-41c1-9aac-9960c648ee87
2026-05-22 12:07:34,111 INFO status has been updated to accepted
2026-05-22 12:07:48,160 INFO status has been updated to running
2026-05-22 12:08:24,487 INFO status has been updated to successful


Ok pour bra 

eyg...


2026-05-22 12:08:40,374 INFO Request ID is b49d7b9f-d871-4b6c-8108-59e4ff5f710b
2026-05-22 12:08:40,595 INFO status has been updated to accepted
2026-05-22 12:08:54,255 INFO status has been updated to running
2026-05-22 12:09:30,634 INFO status has been updated to successful


Ok pour eyg 



# IV - Aggrégation des nouvelles données collectées

Les données collectées se présentent pour le moment sous la forme de fichiers csv enregistrés dans le Drive du projet, à raison d'un fichier par point d'intérêt.

Pour aggréger les nouvelles variables explicatives, on va :

 - Charger les fichiers sous forme de DataFrame Pandas ;
 - Renommer les variables en les faisant commencer par le sigle du point d'intérêt ;
 - Fusionner les jeux de données sur la base de la variable temporelle.


On charge sous forme de DataFrame Pandas les jeux de données qu'on vient de collecter :

In [11]:
# On crée une fonction lambda pour obtenir la date au bon format
# (on conserve la borne de début de l'intervalle de temps)
f = lambda x: pd.to_datetime(x.split('/')[0]).tz_localize("UTC")

In [12]:
# On charge les dataset sous forme de DataFrame pour pouvoir les traiter
# Attention : environ 10 minutes d'exécution...

# On crée un dictionnaire dont les clés seront les préfixes des villes
# et les valeurs les datasets collectés
all_cams = {}

# Pour chaque ville
for ville in df_communes['prefix'] :
    # On affiche où on en est rendu dans les villes
    print(ville + "...")
    
    # Lire les fichiers
    df = pd.read_csv(
        temp_path / ('cams_' + ville + '_' + date_debut + '_' + date_fin + '.csv'),
        sep=';', 
        header=42,
        converters={'# Observation period': f})
    
    # Enregistrer dans all_cams
    all_cams[ville] = df
    
    print(f"{ville} => OK\n")

cru...
cru => OK

sel...
sel => OK

svt...
svt => OK

bra...
bra => OK

eyg...
eyg => OK



On réalise une première observation des jeux de données collectés :

In [13]:
# On commence par regarder à quoi ressemble un dataset cams
display(all_cams['cru'].head())
display(all_cams['cru'].describe())
print(all_cams['cru'].info())


,# Observation period,TOA,Clear sky GHI,Clear sky BHI,Clear sky DHI,Clear sky BNI,GHI,BHI,DHI,BNI,Reliability
0,2019-12-31 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2019-12-31 00:15:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,2019-12-31 00:30:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,2019-12-31 00:45:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,2019-12-31 01:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


,TOA,Clear sky GHI,Clear sky BHI,Clear sky DHI,Clear sky BNI,GHI,BHI,DHI,BNI,Reliability
count,213504.000000,213504.000000,213504.000000,213504.000000,213504.000000,213504.000000,213504.000000,213504.000000,213504.000000,213504.000000
mean,77.202678,56.548935,45.171184,11.377751,85.381012,45.573748,30.262588,15.311159,57.321512,0.978016
std,98.211367,75.535055,63.015503,14.675347,96.094761,66.904949,53.194205,23.439478,83.673502,0.094243
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
50%,2.006150,0.526450,0.038950,0.458750,0.400800,0.352950,0.000000,0.328500,0.000000,1.000000
75%,148.976200,107.474450,85.172825,20.871425,191.531075,79.913575,40.879750,23.167325,121.882725,1.000000
max,308.452700,253.079600,233.125300,112.624400,258.484300,253.079600,233.125300,134.651200,258.484300,1.000000


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 213504 entries, 0 to 213503
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype              
---  ------                --------------   -----              
 0   # Observation period  213504 non-null  datetime64[ns, UTC]
 1   TOA                   213504 non-null  float64            
 2   Clear sky GHI         213504 non-null  float64            
 3   Clear sky BHI         213504 non-null  float64            
 4   Clear sky DHI         213504 non-null  float64            
 5   Clear sky BNI         213504 non-null  float64            
 6   GHI                   213504 non-null  float64            
 7   BHI                   213504 non-null  float64            
 8   DHI                   213504 non-null  float64            
 9   BNI                   213504 non-null  float64            
 10  Reliability           213504 non-null  float64            
dtypes: datetime64[ns, UTC](1), float64(10)
memory usage:

A ce stade, nous avons des données atmosphériques avec une résolution temporelle de **15 minutes**.

Cependant, le jeu de données dans lequel nous souhaitons ajouter ces données atmosphériques a une résolution temporelle de **30 minutes**.

Or les données que nous avons collectées concernant l'atmosphères sont toutes cumulatives (à l'exception de la fiabilité `Reliability`) : pour avoir les données pour 30 minutes nous devons maintenant sommer les lignes deux à deux.

Pour que la variable `Reliability` reste pertinente, on sa remplace par la moyenne entre le temp *t* et le temp *t-1*.

In [14]:
# On parcours l'ensemble des datasets
for df in all_cams.values():
    # Les colonnes à sommer vont de la deuxième à l'avant dernière (index 1 à -1)

    # On sélectionne les heures +15min et +45min (toutes les lignes impaires)
    df_impair = df.iloc[1:-1:2, 1:]
    
    # On sélectionne les heures +0min et +30min (toutes les lignes paires)
    # La première ligne n'est pas traitée et importe peu car a lieu en pleine nuit quand il n'y a pas de soleil
    df_pair = df.iloc[2::2, 1:]

    # On modifie l'index de df_impair pour pouvoir le sommer à df_pair
    df_impair.set_index(df_pair.index, inplace=True)

    # On met à jour le dataset avec les valeurs sommées
    df.iloc[2::2, 1:] = df_pair.add(df_impair)

    # On divise par deux la variable 'Reliability' (dernière colonne) pour avoir sa moyenne
    df.iloc[2::2, -1] /= 2


On renomme les variables des datasets (à l'exeption de la variable `datetime_utc` qui servira à la fusion) en les faisants débuter par le sigle du point d'intérêt correspondant :

In [15]:
for ville, df in all_cams.items() :
    mon_dico = {'# Observation period' : 'datetime_utc',
                'TOA' : ville + '_toa',
                'Clear sky GHI' : ville + '_clear_sky_ghi',
                'Clear sky BHI' : ville + '_clear_sky_bhi',
                'Clear sky DHI' : ville + '_clear_sky_dhi',
                'Clear sky BNI' : ville + '_clear_sky_bni',
                'GHI' : ville + '_ghi',
                'BHI' : ville + '_bhi',
                'DHI' : ville + '_dhi',
                'BNI' : ville + '_bni',
                'Reliability' : ville + '_reliability'}
    df.rename(mon_dico, axis=1, inplace=True)


On fusionne les datasets en se basant sur la variable `datetime_utc` qui contient la date et l'heure des observations :  

In [16]:
for df in all_cams.values():
    df_atmo = df_atmo.merge(right=df, on='datetime_utc', how='left')

On affiche le début du jeu de données pour avoir une première visualisation des nouvelles données :

In [17]:
print(df_atmo.shape)
display(df_atmo.iloc[23:28, :])

(106704, 51)


,datetime_utc,cru_toa,cru_clear_sky_ghi,cru_clear_sky_bhi,cru_clear_sky_dhi,cru_clear_sky_bni,cru_ghi,cru_bhi,cru_dhi,cru_bni,...,eyg_toa,eyg_clear_sky_ghi,eyg_clear_sky_bhi,eyg_clear_sky_dhi,eyg_clear_sky_bni,eyg_ghi,eyg_bhi,eyg_dhi,eyg_bni,eyg_reliability
23,2020-01-01 10:30:00+00:00,251.9616,186.6354,159.4018,27.2336,445.2579,186.6354,159.4018,27.2336,445.2579,...,253.2484,181.5259,150.7824,30.7435,419.0294,181.0210,149.8874,31.1336,416.6206,1.0
24,2020-01-01 11:00:00+00:00,266.4176,199.1656,171.2356,27.9300,452.4020,199.1656,171.2356,27.9300,452.4020,...,268.6971,194.6732,162.9588,31.7144,426.8785,173.7657,126.2110,47.5547,330.7245,1.0
25,2020-01-01 11:30:00+00:00,273.0426,204.8790,176.6443,28.2347,455.3843,204.8790,176.6443,28.2347,455.3843,...,276.2950,201.1038,168.8604,32.2434,430.1928,176.9042,126.1173,50.7869,321.3643,1.0
26,2020-01-01 12:00:00+00:00,271.7235,203.6290,175.4532,28.1758,454.5058,203.6290,175.4532,28.1758,454.5058,...,275.9122,200.6609,168.3135,32.3474,429.3926,177.5202,127.8147,49.7055,326.1265,1.0
27,2020-01-01 12:30:00+00:00,262.4826,195.4154,167.6231,27.7923,449.4843,195.4154,167.6231,27.7923,449.4843,...,267.5549,193.3875,161.3886,31.9990,424.5634,173.1829,126.9076,46.2753,333.8453,1.0


# V - Enregistrement des données CAMS collectées

Nous avons terminé la collecte des données explicatives relatives à l'`atmosphère` pour chaque point significatif du point de vue de la production d'énergie solaire, via le jeu de données **CAMS solar radiation time-serie**.

Nous enregistrons notre jeu de données actuel pour clore la troisième partie de notre travail de collecte.

In [18]:
# On enregistre ce dataset contenant les données de production, astronomiques et météo partielle avant ajout d'autres variables
df_atmo.to_csv(temp_path / "atmosphere_2020_2025.csv", index=False)


In [19]:
# Exemple de la manière dont charger ce dataset final de données atmosphériques :
df_cams = pd.read_csv(temp_path / "atmosphere_2020_2025.csv", parse_dates=['datetime_utc'])
display(df_cams.head())
df_cams.dtypes

,datetime_utc,cru_toa,cru_clear_sky_ghi,cru_clear_sky_bhi,cru_clear_sky_dhi,cru_clear_sky_bni,cru_ghi,cru_bhi,cru_dhi,cru_bni,...,eyg_toa,eyg_clear_sky_ghi,eyg_clear_sky_bhi,eyg_clear_sky_dhi,eyg_clear_sky_bni,eyg_ghi,eyg_bhi,eyg_dhi,eyg_bni,eyg_reliability
0,2019-12-31 23:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2019-12-31 23:30:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,2020-01-01 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,2020-01-01 00:30:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,2020-01-01 01:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


datetime_utc         datetime64[ns, UTC]
cru_toa                          float64
cru_clear_sky_ghi                float64
cru_clear_sky_bhi                float64
cru_clear_sky_dhi                float64
cru_clear_sky_bni                float64
cru_ghi                          float64
cru_bhi                          float64
cru_dhi                          float64
cru_bni                          float64
cru_reliability                  float64
sel_toa                          float64
sel_clear_sky_ghi                float64
sel_clear_sky_bhi                float64
sel_clear_sky_dhi                float64
sel_clear_sky_bni                float64
sel_ghi                          float64
sel_bhi                          float64
sel_dhi                          float64
sel_bni                          float64
sel_reliability                  float64
svt_toa                          float64
svt_clear_sky_ghi                float64
svt_clear_sky_bhi                float64
svt_clear_sky_dh

In [20]:
# Et voici comment concaténer avec les données précédemment collectées :
df_rte = pd.read_csv(temp_path / 'production_2020_2025.csv', parse_dates=['datetime_utc'])
df_astro = pd.read_csv(temp_path / 'astronomie_2020_2025.csv', parse_dates=['datetime_utc'])
df_joint = pd.merge(df_rte, df_astro, on=['datetime_utc'])
df_joint = pd.merge(df_joint, df_cams, on=['datetime_utc'])
print(f"Dimensions des datasets fusionnés : {df_joint.shape}")
display(df_joint)


Dimensions des datasets fusionnés : (106704, 67)


,datetime_utc,consommation,solaire,tco_solaire,tch_solaire,signed_var_tch_tp30,target,cru_azimuth,cru_altitude,sel_azimuth,...,eyg_toa,eyg_clear_sky_ghi,eyg_clear_sky_bhi,eyg_clear_sky_dhi,eyg_clear_sky_bni,eyg_ghi,eyg_bhi,eyg_dhi,eyg_bni,eyg_reliability
0,2019-12-31 23:00:00+00:00,6123.0,0.0,0.0,0.0,0.0,0.0,335.596867,-67.454606,336.734433,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2019-12-31 23:30:00+00:00,5907.0,0.0,0.0,0.0,0.0,0.0,353.817769,-68.883102,354.738150,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,2020-01-01 00:00:00+00:00,5724.0,0.0,0.0,0.0,0.0,0.0,12.883689,-68.564280,13.430816,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,2020-01-01 00:30:00+00:00,5749.0,0.0,0.0,0.0,0.0,0.0,30.261125,-66.570892,30.451783,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,2020-01-01 01:00:00+00:00,5700.0,0.0,0.0,0.0,0.0,0.0,44.631373,-63.284229,44.594428,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106699,2026-01-31 20:30:00+00:00,5573.0,0.0,0.0,0.0,0.0,0.0,286.342039,-40.289762,286.967065,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
106700,2026-01-31 21:00:00+00:00,5491.0,0.0,0.0,0.0,0.0,0.0,293.260657,-45.353013,293.971618,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
106701,2026-01-31 21:30:00+00:00,5514.0,0.0,0.0,0.0,0.0,0.0,301.202197,-50.138700,302.002787,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
106702,2026-01-31 22:00:00+00:00,5618.0,0.0,0.0,0.0,0.0,0.0,310.517465,-54.502032,311.400617,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
